# OLMo 2 7B: Temperature, Retok, and Retok + Temperature

Figure-2-style pass@k curves for the direct joint experiment requested by the reviewer. This notebook compares matched-budget HumanEval, GSM8K, and GSM8K Python results for `allenai/OLMo-2-1124-7B-Instruct` across temperature sampling, retokenization sampling, and temperature decoding from saved retokenized prompt prefixes.

The retok+temperature runs are loaded from `_temp_from_retok` exports. HumanEval and GSM8K Python use structured run directories; GSM8K uses the raw `.df` naming scheme from `experiments/passat/retok/gsm8k_temperature_from_retok.py`. For p=0.0 in the joint family, the notebook uses one random temperature-sampled completion per task.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "figure_notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eval import load, passat

FIGURE_DIR = REPO_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "svg.fonttype": "none",
})


In [ ]:
MODEL_NAME = "allenai/OLMo-2-1124-7B-Instruct"
MODEL_DIR_NAME = MODEL_NAME.replace("/", "_")
MODEL_LABEL = "OLMo 2 7B Instruct"
MAX_K = 50
RETOK_P_VALUES = [0.2, 0.4, 0.6, 0.8, 1.0]
JOINT_P_VALUES = [0.0, *RETOK_P_VALUES]

DATASET_CONFIGS = {
    "humaneval": {
        "label": "HumanEval",
        "dataset_size": 164,
        "numvariants": 51,
        "loader": load.load_humaneval,
        "raw_count_label": "numretokenizations",
    },
    "gsm8k": {
        "label": "GSM8K",
        "dataset_size": 1000,
        "numvariants": 30,
        "loader": load.load_gsm8k,
        "raw_count_label": "numretokenizations",
    },
    "gsm8k_python": {
        "label": "GSM8K Python",
        "dataset_size": 1000,
        "numvariants": 11,
        "loader": load.load_gsm8k_python,
        "raw_count_label": "numretokenizations",
    },
}
DATASETS = ["humaneval", "gsm8k", "gsm8k_python"]

# Match figure_single_model_core_figures.ipynb.
DATASET_COLORS = {
    "humaneval": "blue",
    "gsm8k": "orange",
    "gsm8k_python": "red",
    "mmlu": "green",
}
VARIANT_LINESTYLES = {
    "temperature": "-",
    "retok": "--",
    "retok+temperature": "-.",
}
VARIANT_LABELS = {
    "temperature": "pass@k",
    "retok": "pass@retok",
    "retok+temperature": "pass@retok+temp",
}

RESULT_ROOT = REPO_ROOT / "data" / "raw" / "Tokenizer_passK" / "results_passretok"
print(RESULT_ROOT)


In [ ]:
def normalize_outcomes(df: pd.DataFrame, *, dataset: str) -> pd.DataFrame:
    df = df.copy()
    if "passed" not in df.columns and "Correct" in df.columns:
        df["passed"] = df["Correct"]
    if "passed" not in df.columns and {"generated_answer", "answer"}.issubset(df.columns):
        df["passed"] = (df["generated_answer"] == df["answer"]).astype(int)
    if "passed" not in df.columns:
        raise ValueError("Expected a passed/Correct column or generated_answer+answer columns.")
    if "task_id" not in df.columns and "prompti" in df.columns:
        df["task_id"] = df["prompti"].map(lambda i: f"{dataset}/{int(i)}")
    if "task_id" not in df.columns:
        raise ValueError("Expected task_id or prompti column.")
    df["passed"] = df["passed"].astype(int)
    return df


def model_result_dir(dataset: str) -> Path:
    return RESULT_ROOT / dataset / MODEL_DIR_NAME


def run_dir_for_p(dataset: str, p: float, *, suffix: str = "") -> Path:
    config = DATASET_CONFIGS[dataset]
    return model_result_dir(dataset) / f"retokp_{p}_maxexamples_{config['dataset_size']}_unbiasedsize_{config['numvariants']}{suffix}"


def raw_df_path_for_p(dataset: str, p: float, *, suffix: str = "", sampled_p0: bool = False) -> Path:
    config = DATASET_CONFIGS[dataset]
    filename = (
        f"{dataset}_N_{config['dataset_size']}_{config['raw_count_label']}_"
        f"{config['numvariants']}_{p:.2f}{suffix}.df"
    )
    if p == 0.0 and sampled_p0:
        filename = filename[:-3] + "_sampled.df"
    return model_result_dir(dataset) / filename


def read_result_file(path: Path, *, dataset: str, p: float, family: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix == ".df":
        df = pd.read_hdf(path, key="df")
    else:
        df = pd.read_json(path, lines=True)
    df = normalize_outcomes(df, dataset=dataset)
    df["dataset"] = dataset
    df["p"] = float(p)
    df["family"] = family
    df["source_path"] = str(path)
    return df


def candidate_paths_for_p(dataset: str, p: float, *, suffix: str = "", sampled_p0: bool = False) -> list[Path]:
    return [
        run_dir_for_p(dataset, p, suffix=suffix) / "scored_predictions.jsonl",
        raw_df_path_for_p(dataset, p, suffix=suffix, sampled_p0=sampled_p0),
    ]


def read_first_existing(paths: list[Path], *, dataset: str, p: float, family: str) -> pd.DataFrame | None:
    for path in paths:
        if path.exists():
            return read_result_file(path, dataset=dataset, p=p, family=family)
    return None


def load_temp_from_retok_by_p(dataset: str, p_values=JOINT_P_VALUES) -> dict[float, pd.DataFrame]:
    frames = {}
    missing = []
    for p in p_values:
        paths = candidate_paths_for_p(dataset, p, suffix="_temp_from_retok", sampled_p0=False)
        df = read_first_existing(paths, dataset=dataset, p=p, family="retok+temperature")
        if df is not None:
            frames[p] = df
        else:
            missing.extend(paths)
    if missing:
        print(f"Missing {dataset} retok+temperature files:")
        for path in missing:
            print("  -", path)
    return frames


def load_retok_by_p(dataset: str, p_values=RETOK_P_VALUES) -> dict[float, pd.DataFrame]:
    frames = {}
    missing = []
    for p in p_values:
        paths = candidate_paths_for_p(dataset, p)
        df = read_first_existing(paths, dataset=dataset, p=p, family="retok")
        if df is not None:
            frames[p] = df
        else:
            missing.extend(paths)
    if missing:
        print(f"Missing {dataset} retok files:")
        for path in missing:
            print("  -", path)
    return frames


def try_load_variant(dataset: str, variant_type: str) -> pd.DataFrame | None:
    config = DATASET_CONFIGS[dataset]
    try:
        df = config["loader"](
            model_name=MODEL_NAME,
            dataset_size=int(config["dataset_size"]),
            numvariants=int(config["numvariants"]),
            variant_type=variant_type,
        )
        df = normalize_outcomes(df, dataset=dataset)
        df["dataset"] = dataset
        df["family"] = variant_type
        return df
    except FileNotFoundError as exc:
        print(f"{dataset} {variant_type} baseline was not found by eval.load.")
        print(str(exc).split("\nExpected one of these paths:")[0])
        return None


def one_random_sample_per_task(df: pd.DataFrame, *, random_state: int = 42) -> pd.DataFrame:
    return (
        df.sample(frac=1.0, random_state=random_state)
        .groupby("task_id", sort=False, as_index=False)
        .head(1)
        .sort_values("task_id")
        .reset_index(drop=True)
    )


def curve_for(df: pd.DataFrame, *, max_k: int = MAX_K) -> pd.DataFrame:
    x, y, yerr = passat.pass_curve_points(df, max_k=max_k)
    return pd.DataFrame({"k": x, "pass_rate": y, "pass_rate_std": yerr})


def pass_at_k_value(df: pd.DataFrame, k: int) -> float:
    curve = curve_for(df, max_k=k)
    return float(curve.loc[curve["k"] == k, "pass_rate"].iloc[0])


In [ ]:
dataset_data = {}

for dataset in DATASETS:
    print(f"\n=== {DATASET_CONFIGS[dataset]['label']} ===")
    df_temperature = try_load_variant(dataset, "temperature")
    df_retok_pooled = try_load_variant(dataset, "retok")
    retok_by_p = load_retok_by_p(dataset)
    temp_from_retok_by_p = load_temp_from_retok_by_p(dataset)

    if df_retok_pooled is None:
        print("Falling back to concatenating nonzero-p retok exports; no p=0 greedy row will be included.")
        df_retok_pooled = pd.concat(retok_by_p.values(), ignore_index=True)

    if df_temperature is None and 0.0 in temp_from_retok_by_p:
        df_temperature = temp_from_retok_by_p[0.0].copy()
        df_temperature["family"] = "temperature"
        print("Using p=0.0_temp_from_retok as the temperature-only baseline.")

    joint_frames = []
    if 0.0 in temp_from_retok_by_p:
        p0_joint = one_random_sample_per_task(temp_from_retok_by_p[0.0])
        p0_joint["p"] = 0.0
        p0_joint["family"] = "retok+temperature"
        joint_frames.append(p0_joint)
        print("Using one random p=0.0 temp_from_retok sample per task for retok+temperature.")
    elif df_temperature is not None:
        p0_joint = one_random_sample_per_task(df_temperature)
        p0_joint["p"] = 0.0
        p0_joint["family"] = "retok+temperature"
        if "source_path" not in p0_joint.columns:
            p0_joint["source_path"] = "temperature_baseline_sampled_one_per_task"
        joint_frames.append(p0_joint)
        print("Using one random temperature sample per task as the p=0.0 slice for retok+temperature.")
    else:
        print("No p=0.0 temperature source was found; retok+temperature pooled curve will use nonzero p only.")

    joint_frames.extend(df for p, df in sorted(temp_from_retok_by_p.items()) if p > 0.0)
    df_joint_pooled = pd.concat(joint_frames, ignore_index=True) if joint_frames else pd.DataFrame()
    if not df_joint_pooled.empty:
        df_joint_pooled["dataset"] = dataset
        df_joint_pooled["family"] = "retok+temperature"

    dataset_data[dataset] = {
        "temperature": df_temperature,
        "retok": df_retok_pooled,
        "retok+temperature": df_joint_pooled,
        "retok_by_p": retok_by_p,
        "temp_from_retok_by_p": temp_from_retok_by_p,
    }

    for variant in ["temperature", "retok", "retok+temperature"]:
        df = dataset_data[dataset][variant]
        if df is None or df.empty:
            print(variant, "is empty")
            continue
        counts = df.groupby("task_id").size()
        print(f"{variant}: rows={len(df)}, tasks={df.task_id.nunique()}, samples/task min={counts.min()}, max={counts.max()}")


In [ ]:
records = []
for dataset in DATASETS:
    for variant in ["temperature", "retok", "retok+temperature"]:
        df = dataset_data[dataset][variant]
        if df is None or df.empty:
            continue
        counts = df.groupby("task_id").size()
        n_min = int(counts.min())
        records.append({
            "dataset": dataset,
            "family": variant,
            "tasks": df.task_id.nunique(),
            "rows": len(df),
            "samples_per_task_min": n_min,
            "samples_per_task_max": int(counts.max()),
            "pass@1": pass_at_k_value(df, 1),
            "pass@10": pass_at_k_value(df, min(10, n_min)),
            "pass@50": pass_at_k_value(df, min(MAX_K, n_min)),
        })

summary = pd.DataFrame(records)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
for dataset in DATASETS:
    color = DATASET_COLORS[dataset]
    for variant in ["temperature", "retok", "retok+temperature"]:
        df = dataset_data[dataset][variant]
        if df is None or df.empty:
            continue
        x, y, yerr = passat.pass_curve_points(df, max_k=MAX_K)
        ax.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.05)
        ax.plot(x, y, color=color, linestyle=VARIANT_LINESTYLES[variant], linewidth=2)

dataset_handles = [
    ax.plot([], [], color=DATASET_COLORS[dataset], label=DATASET_CONFIGS[dataset]["label"])[0]
    for dataset in DATASETS
]
variant_handles = [
    ax.plot([], [], color="black", linestyle=VARIANT_LINESTYLES[variant], label=VARIANT_LABELS[variant])[0]
    for variant in ["temperature", "retok", "retok+temperature"]
]
legend_1 = ax.legend(handles=dataset_handles, loc="lower right", frameon=True)
ax.add_artist(legend_1)
ax.legend(handles=variant_handles, loc="lower center", frameon=True)

ax.set_title(f"Pass@k curves: {MODEL_LABEL}")
ax.set_xlim(1, MAX_K)
ax.set_ylim(0.1, 1)
ax.set_xlabel("k")
ax.set_ylabel("Pass Rate")
# ax.set_yscale('log')
ax.grid(True, alpha=0.3)

out = FIGURE_DIR / "olmo2_humaneval_gsm8k_gsm8k_python_temp_retok_joint_pooled_passk.svg"
fig.savefig(out, bbox_inches="tight")
out


In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(5.4 * len(DATASETS), 4.5), constrained_layout=True, sharey=True)
if len(DATASETS) == 1:
    axes = [axes]

for ax, dataset in zip(axes, DATASETS):
    for variant in ["temperature", "retok", "retok+temperature"]:
        df = dataset_data[dataset][variant]
        if df is None or df.empty:
            continue
        p_failure = 1 - df.groupby("task_id").passed.mean().values
        counts, bins = np.histogram(p_failure, bins=np.linspace(0, 1, 25))
        counts = counts / counts.sum()
        ax.step(
            bins,
            np.append(counts, counts[-1]),
            where="post",
            label=VARIANT_LABELS[variant],
            color="black",
            linestyle=VARIANT_LINESTYLES[variant],
        )
        ax.axvline(np.mean(p_failure), color="black", linestyle=VARIANT_LINESTYLES[variant], linewidth=1, alpha=0.6)
    ax.set_title(DATASET_CONFIGS[dataset]["label"])
    ax.set_xlabel(r"$P_{fail}$")
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Fraction of Tasks")
axes[-1].legend(frameon=False)

out = FIGURE_DIR / "olmo2_humaneval_gsm8k_gsm8k_python_temp_retok_joint_pfail.svg"
fig.savefig(out, bbox_inches="tight")
out


In [ ]:
# Optional: inspect exactly which files fed the combined curves.
for dataset in DATASETS:
    print(f"\n=== {DATASET_CONFIGS[dataset]['label']} ===")
    for variant in ["retok", "retok+temperature"]:
        df = dataset_data[dataset][variant]
        print(f"{variant} source paths:")
        if df is None or df.empty or "source_path" not in df.columns:
            print("  - no source_path column")
            continue
        for path in sorted(set(df["source_path"].dropna())):
            print("  -", path)
